In [ ]:
# DRAFT FOR REVIEW: calculation repairs only; no historical rerun yet.
# Historical links, atomic filing selection, and inference remain pending.
#mount drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from review_fixes import (add_next_calendar_return, require_known_returns,
    drawdown_with_initial_capital, annualized_information_ratio,
    winsorize_formation_signal, enhanced_month, validate_evaluation_months)

RETURN_DATA_FILE_NAME =  "/content/drive/MyDrive/FT370/ProjectPart2/data/MonthlyReturn.csv"
LM_DATA_FILE_NAME =  "/content/drive/MyDrive/FT370/ProjectPart2/data/Loughran-McDonald_10X_Summaries_1993-2024.csv"
CIK_DATA_FILE_NAME =  "/content/drive/MyDrive/FT370/ProjectPart2/data/cik-cusip.csv"


In [ ]:
import pandas as pd
import numpy as np

df_return = pd.read_csv(RETURN_DATA_FILE_NAME)
df_lm = pd.read_csv(LM_DATA_FILE_NAME)
df_cik = pd.read_csv(CIK_DATA_FILE_NAME)

print("Monthly return data set columns:")
print(df_return.columns)
print("Loughran-McDonald data set columns:")
print(df_lm.columns)
print("CIK data set columns:")
print(df_cik.columns)

print("\nMonthly return shape:")
print(df_return.shape)
print("Loughran-McDonald shape:")
print(df_lm.shape)
print("CIK shape:")
print(df_cik.shape)

print("\nMonthly return first 5 rows:")
print(df_return.head(5))
print("Loughran-McDonald first 5 rows:")
print(df_lm.head(5))
print("CIK first 5 rows:")
print(df_cik.head(5))

print("\nMonthly return RET data type:")
print(df_return["RET"].dtype)
print("Monthly return date data type:")
print(df_return["date"].dtype)
print("Loughran-McDonald CIK data type:")
print(df_lm["CIK"].dtype)
print("Louhgran-McDonald FILING-DATE data type:")
print(df_lm["FILING_DATE"].dtype)
print("CIK cik data type:")
print(df_cik["cik"].dtype)

print("\nMissing values, monthly return tickers:")
print(df_return["Ticker"].isna().sum())
print("Missing values, monthly return date:")
print(df_return["date"].isna().sum())
print("Missing values, monthly return ret:")
print(df_return["RET"].isna().sum())
print("Missing values, loughran-mcdonald cik:")
print(df_lm["CIK"].isna().sum())
print("Missing values, loughran-mcdonald filing date:")
print(df_lm["FILING_DATE"].isna().sum())
print("Missing values: Loughran-McDonald cik:")
print(df_lm["CIK"].isna().sum())

#Converting returns to numeric values
ret_numeric_check = pd.to_numeric(df_return["RET"], errors="coerce")
print("\nMissing values, monthly return ret before:")
print(df_return["RET"].isna().sum())
print("RET values missing after numeric conversion:")
print(ret_numeric_check.isna().sum())
non_numeric_ret = df_return.loc[ret_numeric_check.isna() & df_return["RET"].notna(), "RET"]
print("Number of non-missing RET values that cannot convert to numeric:")
print(non_numeric_ret.shape[0])

print("\nHow many forms are there?:")
print(df_lm["FORM_TYPE"].value_counts(dropna=False))

print("\nMonthly return date conversion check:")
date_check = pd.to_datetime(df_return["date"], errors="coerce")
print("Original missing monthly return dates:")
print(df_return["date"].isna().sum())
print("Monthly return dates missing after datetime conversion:")
print(date_check.isna().sum())

print("\nLoughran-McDonald FILING_DATE conversion check:")
filing_date_check = pd.to_datetime(df_lm["FILING_DATE"].astype(str), format="%Y%m%d", errors="coerce")
print("Original missing filing dates:")
print(df_lm["FILING_DATE"].isna().sum())
print("Filing dates missing after datetime conversion:")
print(filing_date_check.isna().sum())

print("\nMissing N_Words:")
print(df_lm["N_Words"].isna().sum())
print("Rows where N_Words is 0:")
print((df_lm["N_Words"] == 0).sum())
print("Rows where N_Uncertainty is 0:")
print((df_lm["N_Uncertainty"] == 0).sum())

print("\nLM category missing values:")
lm_category_columns = ["N_Positive","N_Negative","N_Uncertainty","N_Litigious","N_StrongModal","N_WeakModal","N_Constraining"]
print(df_lm[lm_category_columns].isna().sum())

print("\nDuplicate return rows check:")
duplicate_return_rows = df_return.duplicated(subset=["PERMNO", "date"]).sum()
print(duplicate_return_rows)

In [ ]:
df_return_clean = df_return.copy()

df_return_clean["Ticker"] = df_return_clean["Ticker"].str.strip().str.upper()
df_return_clean.loc[df_return["Ticker"].isna(), "Ticker"] = np.nan
print("TIckers that are missing after cleaning")
print(df_return_clean["Ticker"].isna().sum())

df_return_clean["RET"] = pd.to_numeric(df_return_clean["RET"], errors="coerce")
print("RETs that are missing after cleaning")
print(df_return_clean["RET"].isna().sum())

df_return_clean["date"] = pd.to_datetime(df_return_clean["date"], errors="coerce")
print("Dates that are missing after cleaning")
print(df_return_clean["date"].isna().sum())

df_return_clean = df_return_clean.dropna(subset=["Ticker", "RET", "date"]).copy()
print("Mising values after dropping unusable values:")
print(df_return_clean[["Ticker", "date", "RET"]].isna().sum())

df_return_clean["year"] = df_return_clean["date"].dt.year
df_return_clean["month"] = df_return_clean["date"].dt.month
print(df_return_clean.head(5))

print("Checking rows after cleaning:")
print(df_return_clean.shape)

In [ ]:
import requests
import json

#Download SEC ticker-to-CIK mapping
sec_url = "https://www.sec.gov/files/company_tickers.json"

headers = {"User-Agent": "Jeremy Faubert jfaubert@falcon.bentley.edu"}

response = requests.get(sec_url, headers=headers)

sec_data = response.json()

df_sec = pd.DataFrame.from_dict(sec_data, orient="index")
df_sec.columns = ["CIK", "Ticker", "CompanyName"]

df_sec["Ticker"] = df_sec["Ticker"].str.strip().str.upper()

df_sec["CIK"] = df_sec["CIK"].astype(str).str.zfill(10)

print("\nSEC ticker-to-CIK mapping loaded successfully.")
print("SEC mapping shape:")
print(df_sec.shape)

print("\nFirst 5 rows:")
print(df_sec.head(5))

print("\nMissing values:")
print(df_sec[["CIK", "Ticker", "CompanyName"]].isna().sum())

print("\nUnique tickers:")
print(df_sec["Ticker"].nunique())

print("\nUnique CIKs:")
print(df_sec["CIK"].nunique())

In [ ]:
df_return_cik = pd.merge(df_return_clean,df_sec[["Ticker", "CIK", "CompanyName"]],on="Ticker",how="inner")

print("Monthly return rows after SEC merge:")
print(df_return_cik.shape)
print("\nUnique tickers after SEC merge:")
print(df_return_cik["Ticker"].nunique())
print("\nUnique CIKs after SEC merge:")
print(df_return_cik["CIK"].nunique())

In [ ]:
df_lm_clean = df_lm.copy()
df_lm_clean["FORM_TYPE"] = df_lm_clean["FORM_TYPE"].str.strip().str.upper()

main_form_types = ["10-K", "10-Q","10KSB","10QSB","10-K405", "10KSB40","10-KT","10-QT"]
robustness_check_form_types = ["10-K", "10-Q", "10KSB", "10QSB", "10-K405", "10KSB40", "10-KT", "10-QT", "10-K-A", "10-Q-A", "10-K405-A", "10QSB-A", "10KSB-A", "10KSB40-A", "10-KT-A", "10-QT-A"]

df_lm_main = df_lm_clean[df_lm_clean["FORM_TYPE"].isin(main_form_types)].copy()
df_lm_robustness = df_lm_clean[df_lm_clean["FORM_TYPE"].isin(robustness_check_form_types)].copy()

print("Main model shape:")
print(df_lm_main.shape)
print(df_lm_main["FORM_TYPE"].value_counts())

print("Robustness model check shape:")
print(df_lm_robustness.shape)
print(df_lm_robustness["FORM_TYPE"].value_counts())

In [ ]:
df_lm_main["filing_date"] = pd.to_datetime(df_lm_main["FILING_DATE"].astype(str), format="%Y%m%d", errors="coerce")
df_lm_main["year"] = df_lm_main["filing_date"].dt.year
df_lm_main["month"] = df_lm_main["filing_date"].dt.month
print("Missing filing_date after conversion:")
print(df_lm_main["filing_date"].isna().sum())
print(df_lm_main.head(5))

In [ ]:
df_lm_main["Baseline_Sentiment"] = ((df_lm_main["N_Positive"] - df_lm_main["N_Negative"])/ df_lm_main["N_Uncertainty"]) + 1
df_lm_main["Baseline_Sentiment"] = df_lm_main["Baseline_Sentiment"].replace([np.inf, -np.inf], np.nan)
print("Baseline model sentiment missing values:")
print(df_lm_main["Baseline_Sentiment"].isna().sum())

In [ ]:
df_lm_signal = df_lm_main.copy()

df_lm_signal = df_lm_signal[df_lm_signal["N_Words"]> 0].copy()
print(df_lm_signal.shape)

df_lm_signal["positive_rate"] = df_lm_signal["N_Positive"] / df_lm_signal["N_Words"]
df_lm_signal["negative_rate"] = df_lm_signal["N_Negative"] / df_lm_signal["N_Words"]
df_lm_signal["uncertainty_rate"] = df_lm_signal["N_Uncertainty"] / df_lm_signal["N_Words"]
df_lm_signal["litigious_rate"] = df_lm_signal["N_Litigious"] / df_lm_signal["N_Words"]
df_lm_signal["strong_modal_rate"] = df_lm_signal["N_StrongModal"] / df_lm_signal["N_Words"]
df_lm_signal["weak_modal_rate"] = df_lm_signal["N_WeakModal"] / df_lm_signal["N_Words"]
df_lm_signal["constraining_rate"] = df_lm_signal["N_Constraining"] / df_lm_signal["N_Words"]

rate_columns = ["positive_rate", "negative_rate", "uncertainty_rate", "litigious_rate", "strong_modal_rate", "weak_modal_rate", "constraining_rate"]
print(df_lm_signal[rate_columns].isna().sum())

In [ ]:
df_lm_signal = df_lm_signal.sort_values(["CIK", "year", "month", "filing_date"]).copy()
df_lm_monthly = df_lm_signal.drop_duplicates(subset=["CIK", "year", "month"], keep="last").copy()

print(df_lm_monthly.shape)
print(df_lm_monthly[["CIK", "filing_date", "year", "month", "FORM_TYPE", "positive_rate", "negative_rate", "uncertainty_rate", "litigious_rate", "strong_modal_rate", "weak_modal_rate", "constraining_rate", "Baseline_Sentiment"]].head())

In [ ]:
df_model = df_return_cik.copy()
df_model["CIK"] = df_model["CIK"].astype(int)
lm_columns_to_merge = ["CIK", "year", "month", "filing_date", "FORM_TYPE", "Baseline_Sentiment", "positive_rate", "negative_rate", "uncertainty_rate", "litigious_rate", "strong_modal_rate", "weak_modal_rate", "constraining_rate", "FFInd", "SIC"]
df_lm_to_merge = df_lm_monthly[lm_columns_to_merge].copy()

df_model = df_model.merge(df_lm_to_merge, on=["CIK", "year", "month"], how="left")
print(df_model.shape)

print(df_model["Baseline_Sentiment"].notna().sum())
rate_columns = ["positive_rate", "negative_rate", "uncertainty_rate", "litigious_rate", "strong_modal_rate", "weak_modal_rate", "constraining_rate"]
print(df_model[rate_columns].notna().sum())
print(df_model[["PERMNO", "date", "Ticker", "COMNAM", "RET", "CIK", "year", "month", "filing_date", "FORM_TYPE", "Baseline_Sentiment"]].head())

In [ ]:
df_model["month_index"] = (df_model["year"] - df_model["year"].min()) * 12 + df_model["month"]
df_model = df_model.sort_values(["PERMNO", "month_index"]).copy()
columns_to_fill = ["filing_date", "FORM_TYPE", "Baseline_Sentiment", "positive_rate", "negative_rate", "uncertainty_rate", "litigious_rate", "strong_modal_rate", "weak_modal_rate", "constraining_rate", "FFInd", "SIC"]

print(df_model["Baseline_Sentiment"].notna().sum())
print(df_model[rate_columns].notna().sum())

for col in columns_to_fill:
    df_model[col] = df_model.groupby("PERMNO")[col].ffill(limit=3)

print(df_model["Baseline_Sentiment"].notna().sum())
print(df_model[rate_columns].notna().sum())

print(df_model[["PERMNO", "date", "Ticker", "RET", "CIK", "year", "month", "filing_date", "FORM_TYPE", "Baseline_Sentiment", "positive_rate", "negative_rate", "uncertainty_rate"]].head())

In [ ]:
df_model = add_next_calendar_return(df_model)
print("Available exact next-calendar-month returns:", df_model["Next_Month_Return"].notna().sum())
print("Missing or nonadjacent next-calendar-month returns:", df_model["Next_Month_Return"].isna().sum())


In [ ]:
def monthly_z_score(series):
    mean_value = series.mean()
    std_value = series.std()

    if pd.isna(std_value) or std_value == 0:
        return series * np.nan

    return (series - mean_value) / std_value

z_rate_columns = []

for col in rate_columns:
    z_col = "z_" + col

    df_model[z_col] = df_model.groupby(["year", "month"])[col].transform(monthly_z_score)

    z_rate_columns.append(z_col)

df_model["Expanded_LM_Signal"] = (df_model["z_positive_rate"] - df_model["z_negative_rate"] - df_model["z_uncertainty_rate"] - df_model["z_litigious_rate"] - df_model["z_constraining_rate"] + df_model["z_strong_modal_rate"] - df_model["z_weak_modal_rate"])

print(z_rate_columns)

print(df_model["Expanded_LM_Signal"].notna().sum())

print(df_model[["PERMNO", "date", "Ticker", "RET", "Next_Month_Return", "Expanded_LM_Signal", "z_positive_rate", "z_negative_rate", "z_uncertainty_rate", "z_litigious_rate", "z_constraining_rate", "z_strong_modal_rate", "z_weak_modal_rate"]].sample())

In [ ]:
backtest_columns = ["PERMNO", "date", "Ticker", "COMNAM", "CIK", "year", "month", "month_index", "RET", "Next_Month_Return", "Baseline_Sentiment", "Expanded_LM_Signal", "FFInd", "SIC"]
df_backtest = df_model[backtest_columns].copy()

print(df_backtest.shape)

print(df_backtest[["PERMNO", "date", "Ticker", "CIK", "Next_Month_Return", "Baseline_Sentiment", "Expanded_LM_Signal", "FFInd"]].isna().sum())
print(df_backtest.head())

In [ ]:
baseline_results = []

for year in range(1994, 2024):
  for month in range(1,13):
    month_data = df_backtest[(df_backtest["year"] == year) & (df_backtest["month"] == month)].copy()
    month_data = month_data[["PERMNO", "Ticker", "Baseline_Sentiment", "Next_Month_Return"]].dropna(subset=["Baseline_Sentiment"])

    if len(month_data) == 0:
      continue
    require_known_returns(month_data, f"Baseline {year}-{month:02d}")
    benchmark_return = month_data["Next_Month_Return"].mean()
    long_data = month_data[month_data["Baseline_Sentiment"] > 0].copy()

    if len(long_data) > 0:
      long_data["weight"] = (long_data["Baseline_Sentiment"]/long_data["Baseline_Sentiment"].abs().sum())
      long_return = (long_data["weight"] * long_data["Next_Month_Return"]).sum()
    else:
      long_return = np.nan

    short_data = month_data[month_data["Baseline_Sentiment"] < 0].copy()

    if len(short_data) > 0:
      short_data["weight"] = (short_data["Baseline_Sentiment"]/short_data["Baseline_Sentiment"].abs().sum())
      short_return = (short_data["weight"] * short_data["Next_Month_Return"]).sum()
    else:
      short_return = np.nan

    if pd.notna(long_return) and pd.notna(short_return):
      long_short_return = long_return + short_return
    else:
      long_short_return = np.nan

    baseline_results.append({"year": year, "month": month, "benchmark_return": benchmark_return, "long_return": long_return, "short_return": short_return, "long_short_return": long_short_return, "num_firms": len(month_data), "num_long": len(long_data), "num_short": len(short_data)})

baseline_results = pd.DataFrame(baseline_results)

print(baseline_results.shape)
print(baseline_results.head())

print("Missing baseline long-short returns:")
print(baseline_results["long_short_return"].isna().sum())

print("\nBaseline results summary:")
print(baseline_results[["benchmark_return", "long_return", "short_return", "long_short_return", "num_firms", "num_long", "num_short"]].describe())

In [ ]:
main_results = []

for year in range(1994, 2024):
  for month in range(1, 13):

    month_data = df_backtest[(df_backtest["year"] == year) & (df_backtest["month"] == month)].copy()

    month_data = month_data[["PERMNO", "Ticker", "Expanded_LM_Signal", "Next_Month_Return"]].dropna(subset=["Expanded_LM_Signal"])

    if len(month_data) < 5:
      continue

    require_known_returns(month_data, f"Expanded {year}-{month:02d}")
    benchmark_return = month_data["Next_Month_Return"].mean()

    month_data["signal_rank"] = month_data["Expanded_LM_Signal"].rank(method="first")

    month_data["quintile"] = pd.qcut(month_data["signal_rank"],q=5, labels=["F1", "F2", "F3", "F4", "F5"])

    quintile_returns = month_data.groupby("quintile", observed=False)["Next_Month_Return"].mean()

    f1_return = quintile_returns.loc["F1"]
    f2_return = quintile_returns.loc["F2"]
    f3_return = quintile_returns.loc["F3"]
    f4_return = quintile_returns.loc["F4"]
    f5_return = quintile_returns.loc["F5"]

    long_short_return = f5_return - f1_return

    main_results.append({"year": year, "month": month, "benchmark_return": benchmark_return, "F1_return": f1_return, "F2_return": f2_return, "F3_return": f3_return, "F4_return": f4_return, "F5_return": f5_return, "long_short_return": long_short_return, "F5_minus_benchmark": f5_return - benchmark_return, "benchmark_minus_F1": benchmark_return - f1_return, "num_firms": len(month_data), "F1_count": (month_data["quintile"] == "F1").sum(), "F2_count": (month_data["quintile"] == "F2").sum(), "F3_count": (month_data["quintile"] == "F3").sum(), "F4_count": (month_data["quintile"] == "F4").sum(), "F5_count": (month_data["quintile"] == "F5").sum()})

main_results = pd.DataFrame(main_results)

print("Main expanded LM backtest shape:")
print(main_results.shape)

print("\nFirst 5 rows:")
print(main_results.head())

print("\nMissing long-short returns:")
print(main_results["long_short_return"].isna().sum())

print("\nMain expanded LM summary:")
print(main_results[["benchmark_return", "F1_return", "F2_return", "F3_return", "F4_return", "F5_return", "long_short_return", "F5_minus_benchmark", "benchmark_minus_F1", "num_firms"]].describe())

In [ ]:
df_enhanced = df_backtest.copy()
# Only formation-time signals are winsorized. Realized returns are unchanged.
df_enhanced["Expanded_LM_Signal_Winsorized"] = (
    df_enhanced.groupby(["year", "month"])["Expanded_LM_Signal"]
    .transform(winsorize_formation_signal)
)
assert df_enhanced["Next_Month_Return"].equals(df_backtest["Next_Month_Return"])
print("Enhanced payoffs retain the original realized returns.")


In [ ]:
enhanced_results = []
for year in range(1994, 2024):
    for month in range(1, 13):
        month_data = df_enhanced[(df_enhanced["year"] == year) & (df_enhanced["month"] == month)].copy()
        if month_data.empty:
            continue
        enhanced_results.append(enhanced_month(month_data, year, month, cap=0.05))
enhanced_results = pd.DataFrame(enhanced_results)
print(enhanced_results.head())
print("Maximum position weight:", enhanced_results["max_name_weight"].max())
print("Months with partial allocation:", (enhanced_results["long_allocated"] < 1 - 1e-10).sum())
print("Industry budgets match across long and short sleeves; infeasible capacity stays unallocated.")


In [ ]:
def calculate_strategy_metrics(strategy_name, results_df, signal_data, signal_column, return_column):
  metrics = {}

  strategy_returns = pd.to_numeric(results_df["long_short_return"], errors="coerce")
  if strategy_returns.isna().any() or not np.isfinite(strategy_returns).all():
    raise ValueError("Resolve every monthly portfolio outcome before reporting performance.")
  if (strategy_returns < -1).any():
    raise ValueError("Portfolio insolvency requires explicit accounting; compounding stopped.")
  validate_evaluation_months(results_df)

  metrics["Average Monthly Long-Short Return"] = strategy_returns.mean()

  metrics["Information Ratio"] = annualized_information_ratio(
      strategy_returns, results_df.loc[strategy_returns.index, "benchmark_return"])
  # IR is now ANNUALIZED against the supplied universe benchmark.
  # This benchmark definition still needs a common-universe review.

  metrics["Hit Rate"] = (strategy_returns > 0).mean()

  total_months = len(strategy_returns)
  cumulative_wealth = (1 + strategy_returns).prod()
  metrics["Annualized Return"] = cumulative_wealth ** (12 / total_months) - 1

  drawdown = drawdown_with_initial_capital(strategy_returns)
  metrics["Maximum Drawdown"] = drawdown.min()

  metrics["Sharpe Ratio"] = (strategy_returns.mean() / strategy_returns.std()) * np.sqrt(12)

  metrics["Cumulative Return"] = cumulative_wealth - 1

  backtest_months = results_df[["year", "month"]].drop_duplicates().copy()

  ic_values = []

  ic_data = signal_data[["year", "month", signal_column, return_column]].dropna().copy()
  ic_data = ic_data.merge(backtest_months, on=["year", "month"], how="inner")

  for (year, month), month_data in ic_data.groupby(["year", "month"]):
    if month_data[signal_column].nunique() > 1 and month_data[return_column].nunique() > 1:
      ic = month_data[signal_column].corr(month_data[return_column])
      ic_values.append(ic)

  metrics["Average IC"] = np.mean(ic_values)

  metrics["IC Months"] = len(ic_values)

  metrics_df = pd.DataFrame({"Strategy": strategy_name, "Metric": list(metrics.keys()), "Value": list(metrics.values())})

  return metrics_df

In [ ]:
baseline_metrics = calculate_strategy_metrics(strategy_name="Baseline", results_df=baseline_results, signal_data=df_backtest, signal_column="Baseline_Sentiment", return_column="Next_Month_Return")

main_metrics = calculate_strategy_metrics(strategy_name="Expanded LM Quintiles", results_df=main_results, signal_data=df_backtest, signal_column="Expanded_LM_Signal", return_column="Next_Month_Return")

enhanced_metrics = calculate_strategy_metrics(strategy_name="Enhanced Sector-Neutral Capped", results_df=enhanced_results, signal_data=df_enhanced, signal_column="Expanded_LM_Signal_Winsorized", return_column="Next_Month_Return")

all_metrics = pd.concat([baseline_metrics, main_metrics, enhanced_metrics], ignore_index=True)

print("All strategy performance metrics:")
print(all_metrics)

In [ ]:
all_metrics_clean = all_metrics.copy()

percent_metrics = ["Average Monthly Long-Short Return", "Hit Rate", "Annualized Return", "Maximum Drawdown", "Cumulative Return"]

all_metrics_clean["Formatted Value"] = all_metrics_clean.apply(lambda row: f"{row['Value']:.2%}" if row["Metric"] in percent_metrics else f"{row['Value']:.4f}", axis=1)

print(all_metrics_clean[["Strategy", "Metric", "Formatted Value"]])

In [ ]:
import matplotlib.pyplot as plt
# Dates label the holding-return month; starting capital is plotted explicitly.
def make_wealth_plot(results):
    dates = pd.to_datetime(results[["year", "month"]].assign(day=1)) + pd.offsets.MonthEnd(2)
    body = pd.DataFrame({"date": dates, "wealth_index": (1 + results["long_short_return"]).cumprod()})
    initial = pd.DataFrame({"date": [dates.iloc[0] - pd.offsets.MonthEnd(1)], "wealth_index": [1.0]})
    return pd.concat([initial, body], ignore_index=True)
baseline_plot = make_wealth_plot(baseline_results)
main_plot = make_wealth_plot(main_results)
enhanced_plot = make_wealth_plot(enhanced_results)
colors = {"Baseline": "tab:blue", "Expanded LM Quintiles": "tab:orange", "Enhanced": "tab:green"}
series = [("Baseline", baseline_plot), ("Expanded LM Quintiles", main_plot), ("Enhanced", enhanced_plot)]
for subset, use_log in [(series, True), (series[1:], False)]:
    plt.figure(figsize=(12, 6))
    for name, frame in subset:
        plt.plot(frame["date"], frame["wealth_index"], label=name, color=colors[name])
    if use_log:
        plt.yscale("log")
    plt.title("DRAFT: cumulative portfolio spread, before costs")
    plt.xlabel("Holding-return month (calendar month-end label)")
    plt.ylabel("Wealth index, including initial capital of 1")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
main_fractile_means = main_results[["F1_return", "F2_return", "F3_return", "F4_return", "F5_return"]].mean()

plt.figure(figsize=(8, 5))
plt.bar(["F1", "F2", "F3", "F4", "F5"], main_fractile_means.values)
plt.title("Average Monthly Returns by Expanded LM Signal Quintile")
plt.xlabel("Signal Quintile")
plt.ylabel("Average Monthly Return")
plt.gca().yaxis.set_major_formatter(__import__("matplotlib.ticker", fromlist=["PercentFormatter"]).PercentFormatter(1.0))
plt.grid(axis="y")
plt.show()

In [ ]:
enhanced_fractile_means = enhanced_results[["F1_return", "F2_return", "F3_return", "F4_return", "F5_return"]].mean()

plt.figure(figsize=(8, 5))
plt.bar(["F1", "F2", "F3", "F4", "F5"], enhanced_fractile_means.values)
plt.title("Equal-weight returns by within-industry signal quintile")
plt.xlabel("F1 = low signal; F5 = high signal")
plt.ylabel("Average Monthly Return")
plt.gca().yaxis.set_major_formatter(__import__("matplotlib.ticker", fromlist=["PercentFormatter"]).PercentFormatter(1.0))
plt.grid(axis="y")
plt.show()

In [ ]:
import statsmodels.formula.api as smf

df_regression = df_backtest.copy()

df_regression = df_regression[(df_regression["year"] >= 1994) & (df_regression["year"] <= 2023)].copy()

df_regression["year_month"] = df_regression["year"].astype(str) + "-" + df_regression["month"].astype(str).str.zfill(2)

baseline_regression_data = df_regression[["Next_Month_Return", "Baseline_Sentiment", "year_month"]].dropna().copy()
expanded_regression_data = df_regression[["Next_Month_Return", "Expanded_LM_Signal", "year_month"]].dropna().copy()

baseline_regression = smf.ols("Next_Month_Return ~ Baseline_Sentiment + C(year_month)", data=baseline_regression_data).fit()
expanded_regression = smf.ols("Next_Month_Return ~ Expanded_LM_Signal + C(year_month)", data=expanded_regression_data).fit()

regression_summary = pd.DataFrame({
    "Model": ["Baseline", "Expanded LM Signal"],
    "Signal Coefficient": [baseline_regression.params["Baseline_Sentiment"], expanded_regression.params["Expanded_LM_Signal"]],
    "t-stat": [baseline_regression.tvalues["Baseline_Sentiment"], expanded_regression.tvalues["Expanded_LM_Signal"]],
    "p-value": [baseline_regression.pvalues["Baseline_Sentiment"], expanded_regression.pvalues["Expanded_LM_Signal"]],
    "Adjusted R-squared": [baseline_regression.rsquared_adj, expanded_regression.rsquared_adj],
    "Observations": [baseline_regression.nobs, expanded_regression.nobs]
})

print(regression_summary)

In [ ]:
metrics_table = all_metrics.pivot(index="Strategy", columns="Metric", values="Value").reset_index()

metrics_table = metrics_table[["Strategy", "Average Monthly Long-Short Return", "Information Ratio", "Hit Rate", "Annualized Return", "Maximum Drawdown", "Sharpe Ratio", "Cumulative Return", "Average IC", "IC Months"]]

metrics_table_formatted = metrics_table.copy()

percent_columns = ["Average Monthly Long-Short Return", "Hit Rate", "Annualized Return", "Maximum Drawdown", "Cumulative Return"]

for col in percent_columns:
  metrics_table_formatted[col] = metrics_table_formatted[col].map(lambda x: f"{x:.2%}")

decimal_columns = ["Information Ratio", "Sharpe Ratio", "Average IC"]

for col in decimal_columns:
  metrics_table_formatted[col] = metrics_table_formatted[col].map(lambda x: f"{x:.4f}")

metrics_table_formatted["IC Months"] = metrics_table_formatted["IC Months"].astype(int)

print(metrics_table_formatted)

In [ ]:
fractile_rows = []
for name, frame, weighted in [
    ("Expanded LM Quintiles", main_results, False),
    ("Enhanced Industry-Neutral Capped", enhanced_results, True),
]:
    row = {"Strategy": name}
    row.update({q: frame[q + "_return"].mean() for q in ["F1", "F2", "F3", "F4", "F5"]})
    row["Equal-weight F5 - F1"] = (frame["F5_return"] - frame["F1_return"]).mean()
    row["Portfolio short-underlying sleeve"] = frame["weighted_F1_return" if weighted else "F1_return"].mean()
    row["Portfolio long sleeve"] = frame["weighted_F5_return" if weighted else "F5_return"].mean()
    row["Portfolio spread"] = frame["long_short_return"].mean()
    fractile_rows.append(row)
fractile_table = pd.DataFrame(fractile_rows)
fractile_table_formatted = fractile_table.copy()
for col in fractile_table.columns.drop("Strategy"):
    fractile_table_formatted[col] = fractile_table[col].map(lambda x: f"{x:.2%}")
print(fractile_table_formatted)
print("Equal-weight diagnostics may omit empty-industry months; portfolio spreads include flat months.")


In [ ]:
regression_table = regression_summary.copy()

regression_table["Signal Coefficient"] = regression_table["Signal Coefficient"].map(lambda x: f"{x:.6f}")
regression_table["t-stat"] = regression_table["t-stat"].map(lambda x: f"{x:.3f}")
regression_table["p-value"] = regression_table["p-value"].map(lambda x: f"{x:.6f}")
regression_table["Adjusted R-squared"] = regression_table["Adjusted R-squared"].map(lambda x: f"{x:.4f}")
regression_table["Observations"] = regression_table["Observations"].astype(int)

print(regression_table)